In [ ]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image, ImageDraw, ImageFont

from utils.mesh_utils import *

In [ ]:
# --- User-editable section ---

# Option 2: leave `colors` undefined to use a generated demo palette
# -------------------------------


def luminance(rgb):
    """Rec. 709 luminance approximation from linear rgb in [0,1]."""
    r, g, b = rgb
    return 0.2126 * r + 0.7152 * g + 0.0722 * b

def create_color_grid_image(colors,
                            cols=20, rows=25,
                            cell_size=64,
                            margin=2,
                            out_path="color_grid.png"):
    arr = np.array(colors, dtype=float)
    arr = arr.reshape((rows, cols, 3))

    # image dimensions
    W = cols * cell_size + 2 * margin
    H = rows * cell_size + 2 * margin
    img = Image.new("RGB", (W, H), (255, 255, 255))
    draw = ImageDraw.Draw(img)

    font = ImageFont.load_default()

    for r in range(rows):
        for c in range(cols):
            idx = r * cols + c
            rgb = arr[r, c]
            # convert to 0-255 ints
            color = tuple(int(round(x * 255)) for x in rgb)
            x0 = margin + c * cell_size
            y0 = margin + r * cell_size
            x1 = x0 + cell_size
            y1 = y0 + cell_size

            draw.rectangle([x0, y0, x1, y1], fill=color)

            txt = f"{idx}"

            lum = luminance(rgb)
            text_color = (255, 255, 255) if lum < 0.5 else (0, 0, 0)

            bbox = draw.multiline_textbbox((0,0), txt, font=font, align="center")
            tw = bbox[2] - bbox[0]
            th = bbox[3] - bbox[1]

            tx = x0 + (cell_size - tw) / 2
            ty = y0 + (cell_size - th) / 2

            # draw text (multiline)
            draw.multiline_text((tx, ty), txt, font=font, fill=text_color, align="center")

    # optional: draw a thin grid line (uncomment if desired)
    # for r in range(rows + 1):
    #     y = margin + r * cell_size
    #     draw.line([(margin, y), (W - margin - 1, y)], fill=(200,200,200))
    # for c in range(cols + 1):
    #     x = margin + c * cell_size
    #     draw.line([(x, margin), (x, H - margin - 1)], fill=(200,200,200))

    img.save(out_path)
    print(f"Saved color grid to: {out_path}  (size: {W}x{H} pixels)")

create_color_grid_image(colors[:100], cols=10, rows=10, cell_size=64, out_path="colors_grid.png")

In [ ]:
patch_path = f"example_results/vsa/2_merged_proxy_cluster.txt"
patch_outpath = "example_results/vsa/2_pre_proxy_cluster.txt"

patch2f = read_patch_cluster(patch_path)

out_patch2f = []
merged0 = []
merged1 = []
for i in range(len(patch2f)):
    if i not in [5,6,7,9,10,11,12,13]:
        out_patch2f.append(patch2f[i])
    else:
        if i in [5, 6, 7]:
            merged0.extend(patch2f[i])
            if i == 7:
                out_patch2f.append(merged0)
        elif i in [9, 10, 11, 12, 13]:
            merged1.extend(patch2f[i])
            if i == 13:
                out_patch2f.append(merged1)
write_proxy_cluster(out_patch2f, patch_outpath)
    


In [ ]:
def mesh2voxel(mesh_file):
    mesh_name = os.path.basename(mesh_file).split('.')[0]
    mesh_path = os.path.join(os.path.dirname(mesh_file), mesh_name)
    mesh = o3d.io.read_triangle_mesh(mesh_file)
    
    mesh.scale(1 / np.max(mesh.get_max_bound() - mesh.get_min_bound()), center=mesh.get_center())
    o3d.visualization.draw_geometries([mesh])
    
    print('voxelization')
    voxel_grid = o3d.geometry.VoxelGrid.create_from_triangle_mesh(mesh, voxel_size=0.005)
    o3d.visualization.draw_geometries([voxel_grid])
    
    output_file = os.path.join(mesh_path + '.voxel')
    o3d.io.write_voxel_grid(output_file, voxel_grid)
    
path = 'example_data/example_CAD/2.obj'
mesh2voxel(path)